In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import load_model
import pickle
import serial
import time
import keyboard  # Instala el paquete 'keyboard' si aún no lo tienes: pip install keyboard

# Establecer conexión serial con Arduino
try:
    arduino = serial.Serial('COM6', 115200)  # Cambia 'COM6' por el puerto al que está conectado tu Arduino
    print("Conexión establecida con Arduino.")
except serial.SerialException as e:
    print(f"No se pudo establecer conexión con Arduino: {e}")
    exit()

# Cargar el modelo entrenado
try:
    modelo_cargado = load_model('modelo_red_neuronal_reducido.keras')
    print("Modelo cargado exitosamente.")
except Exception as e:
    print(f"Error al cargar el modelo: {e}")
    arduino.close()
    exit()

# Cargar el StandardScaler guardado
try:
    with open('standard_scaler.pkl', 'rb') as scaler_file:
        scaler = pickle.load(scaler_file)
    print("Scaler cargado exitosamente.")
except FileNotFoundError as e:
    print(f"No se encontró el archivo de escalado: {e}")
    arduino.close()
    exit()

# Tiempo de captura en segundos
tiempo_captura = 1  # Capturar datos durante 1 segundo

while True:
    # Tiempo inicial de captura
    tiempo_inicio = time.time()

    # Leer datos desde Arduino durante el tiempo especificado
    datos = []
    while (time.time() - tiempo_inicio) < tiempo_captura:
        try:
            linea = arduino.readline().decode().strip()
            if linea:
                datos.append(float(linea))
        except ValueError as e:
            print(f"Error al convertir los datos: {e}")
            continue

    if not datos:
        print("No se recibieron datos.")
        continue

    # Crear DataFrame con los datos capturados
    df_nuevos_datos = pd.DataFrame({'Dato': datos})

    # Calcular características de los nuevos datos
    N = len(df_nuevos_datos["Dato"])
    MAV = np.mean(np.abs(df_nuevos_datos["Dato"]))
    VAR = np.var(df_nuevos_datos["Dato"])
    RMS = np.sqrt(np.mean(df_nuevos_datos["Dato"] ** 2))
    Wav_Leng = np.sum(np.abs(np.diff(df_nuevos_datos["Dato"])))
    diff_signal = np.diff(df_nuevos_datos["Dato"])
    Dasdv = np.sqrt(np.mean(diff_signal ** 2))
    mean_value = np.mean(df_nuevos_datos["Dato"])
    Damv = np.mean(np.abs(df_nuevos_datos["Dato"] - mean_value))
    iav_value = np.sum(np.abs(df_nuevos_datos["Dato"]))

    # Crear DataFrame con las características calculadas
    df_caracteristicas = pd.DataFrame({
        'MAV_total': [MAV],
        'VAR_total': [VAR],
        'RMS_total': [RMS],
        'Wav_Leng': [Wav_Leng],
        'Dasdv': [Dasdv],
        'Damv': [Damv],
        'iav': [iav_value]
    })

    # Aplicar factores de escalado (actualmente 1.0)
    factores_escalado = {
        "MAV_total": 1.0,
        "VAR_total": 1.0,
        "RMS_total": 1.0,
        "Wav_Leng": 1.0,
        "Dasdv": 1.0,
        "Damv": 1.0,
        "iav": 1.0
    }

    for caracteristica, factor in factores_escalado.items():
        df_caracteristicas[caracteristica] = df_caracteristicas[caracteristica] * factor

    # Escalar los datos utilizando el mismo escalador que se usó durante el entrenamiento
    X_new = scaler.transform(df_caracteristicas)

    # Realizar predicciones con el modelo cargado
    y_predicciones = modelo_cargado.predict(X_new)

    # Definir las etiquetas de clases
    etiquetas = ['Mano abierta', 'Mano pinza', 'Dedo anular', 'Dedo indice']

    # Obtener la clase predicha (la clase con la probabilidad más alta)
    clase_predicha = np.argmax(y_predicciones, axis=1)[0]

    # Enviar la clase predicha por serial
    # Aquí definimos la lógica que envía los valores según la clase predicha.
    if clase_predicha == 0:
        arduino.write("0\n".encode())  # Envía '0' por serial si la clase es "Mano abierta"
    elif clase_predicha == 1:
        arduino.write("1\n".encode())  # Envía '1' por serial si la clase es "Mano pinza"
    elif clase_predicha == 2:
        arduino.write("2\n".encode())  # Envía '2' por serial si la clase es "Dedo anular"
    elif clase_predicha == 3:
        arduino.write("3\n".encode())  # Envía '3' por serial si la clase es "Dedo corazon"

    # Mostrar la predicción
    print(f"Predicción (clase): {clase_predicha}, Predicción (etiqueta): {etiquetas[clase_predicha]}")

    # Verificar si se presionó la tecla 'Esc' para salir del bucle
    if keyboard.is_pressed('Esc'):
        print("Saliendo del programa...")
        break

# Cerrar la conexión serial con Arduino al finalizar
arduino.close()

Conexión establecida con Arduino.
Modelo cargado exitosamente.
Scaler cargado exitosamente.


C:\Users\david\miniconda3\envs\entornoML2024\lib\site-packages\sklearn\base.py:347: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.0 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step
Predicción (clase): 3, Predicción (etiqueta): Dedo indice
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
Predicción (clase): 0, Predicción (etiqueta): Mano abierta
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Predicción (clase): 3, Predicción (etiqueta): Dedo indice
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Predicción (clase): 3, Predicción (etiqueta): Dedo indice
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
Predicción (clase): 0, Predicción (etiqueta): Mano abierta
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
Predicción (clase): 3, Predicción (etiqueta): Dedo indice
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
Predicción (clase): 3, Predicción (etiqueta): Dedo indice
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
Predicción (clase): 3, Predicción (etiqueta): Dedo indice
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
Predicción (clase): 0, Predicción (etiqueta): Mano abierta
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
Predicción (clase): 0, Predicción (etiqueta): Mano abierta
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/st

In [ ]:
import tkinter as tk
import threading
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import load_model
import pickle
import serial
import time

# ---- Configuración inicial ----
# Conectar con el ESP32 (cambia 'COM6' por tu puerto)
try:
    arduino = serial.Serial('COM6', 115200)
    print("Conexión establecida con el ESP32.")
except serial.SerialException as e:
    print(f"No se pudo conectar con el ESP32: {e}")
    exit()

# Cargar el modelo entrenado
try:
    modelo_cargado = load_model('modelo_red_neuronal_reducido.keras')
    print("Modelo cargado exitosamente.")
except Exception as e:
    print(f"Error al cargar el modelo: {e}")
    arduino.close()
    exit()

# Cargar el StandardScaler guardado
try:
    with open('standard_scaler.pkl', 'rb') as scaler_file:
        scaler = pickle.load(scaler_file)
    print("Scaler cargado exitosamente.")
except FileNotFoundError as e:
    print(f"No se encontró el archivo de escalado: {e}")
    arduino.close()
    exit()

# Variable de control del bucle de captura
capturando = False

# ---- Función principal de captura en bucle ----
def ejecutar_captura():
    """Captura datos en un bucle infinito hasta que se presione 'Detener'."""
    global capturando
    capturando = True
    while capturando:
        datos = []

        # Capturar 100 datos
        for _ in range(100):
            if not capturando:  # Si se presiona detener, salir del bucle
                return
            try:
                linea = arduino.readline().decode().strip()
                if linea:
                    datos.append(float(linea))
            except ValueError:
                continue
        
        if len(datos) < 100:
            continue  # Si no se capturaron suficientes datos, repetir

        # Calcular características
        df_nuevos_datos = pd.DataFrame({'Dato': datos})
        MAV = np.mean(np.abs(df_nuevos_datos["Dato"]))
        VAR = np.var(df_nuevos_datos["Dato"])
        RMS = np.sqrt(np.mean(df_nuevos_datos["Dato"] ** 2))
        Wav_Leng = np.sum(np.abs(np.diff(df_nuevos_datos["Dato"])))
        Dasdv = np.sqrt(np.mean(np.diff(df_nuevos_datos["Dato"]) ** 2))
        mean_value = np.mean(df_nuevos_datos["Dato"])
        Damv = np.mean(np.abs(df_nuevos_datos["Dato"] - mean_value))
        iav_value = np.sum(np.abs(df_nuevos_datos["Dato"]))

        # Crear DataFrame con características
        df_caracteristicas = pd.DataFrame({
            'MAV_total': [MAV],
            'VAR_total': [VAR],
            'RMS_total': [RMS],
            'Wav_Leng': [Wav_Leng],
            'Dasdv': [Dasdv],
            'Damv': [Damv],
            'iav': [iav_value]
        })

        # Escalar características
        X_new = scaler.transform(df_caracteristicas)

        # Realizar predicción
        y_predicciones = modelo_cargado.predict(X_new)
        etiquetas = ['Mano abierta', 'Mano pinza', 'Dedo anular', 'Dedo indice']
        clase_predicha = np.argmax(y_predicciones, axis=1)[0]
        resultado = etiquetas[clase_predicha]

        # Enviar predicción al ESP32
        arduino.write(f"{clase_predicha}\n".encode())

        # Mostrar en la interfaz
        etiqueta_prediccion.config(text=f"Última predicción: {resultado}")

        # Pequeña pausa antes de la siguiente iteración
        time.sleep(0.5)

def iniciar_captura():
    """Inicia el bucle de captura en un hilo separado."""
    global capturando
    if not capturando:
        threading.Thread(target=ejecutar_captura, daemon=True).start()

def detener_captura():
    """Detiene la captura pero mantiene la conexión abierta."""
    global capturando
    capturando = False

def enviar_movimiento(movimiento):
    """Envía un comando específico al ESP32."""
    comandos = {
        "Mano Abierta": "0",
        "Mano Cerrada": "1",
        "Anular": "2",
        "Índice": "3"
    }
    arduino.write(f"{comandos[movimiento]}\n".encode())
    etiqueta_prediccion.config(text=f"Enviado: {movimiento}")

# ---- Interfaz gráfica ----
ventana = tk.Tk()
ventana.title("Interfaz de Control")
ventana.geometry("400x400")

# ---- Botones de control ----
lbl_modo_captura = tk.Label(ventana, text="Control de Captura", font=("Arial", 12))
lbl_modo_captura.pack(pady=10)

btn_iniciar_captura = tk.Button(ventana, text="Iniciar Captura", font=("Arial", 10), command=iniciar_captura, bg="green")
btn_iniciar_captura.pack(pady=5)

btn_detener_captura = tk.Button(ventana, text="Detener Captura", font=("Arial", 10), command=detener_captura, bg="red")
btn_detener_captura.pack(pady=5)

# ---- Mostrar predicción ----
etiqueta_prediccion = tk.Label(ventana, text="Última predicción: ---", font=("Arial", 12))
etiqueta_prediccion.pack(pady=10)

# ---- Botones para movimientos predeterminados ----
lbl_movimientos = tk.Label(ventana, text="Movimientos Predeterminados", font=("Arial", 12))
lbl_movimientos.pack(pady=10)

btn_mano_abierta = tk.Button(ventana, text="Mano Abierta", font=("Arial", 10), command=lambda: enviar_movimiento("Mano Abierta"))
btn_mano_abierta.pack(pady=5)

btn_mano_cerrada = tk.Button(ventana, text="Mano Cerrada", font=("Arial", 10), command=lambda: enviar_movimiento("Mano Cerrada"))
btn_mano_cerrada.pack(pady=5)

btn_pinza = tk.Button(ventana, text="Anular", font=("Arial", 10), command=lambda: enviar_movimiento("Anular"))
btn_pinza.pack(pady=5)

btn_indice = tk.Button(ventana, text="Índice", font=("Arial", 10), command=lambda: enviar_movimiento("Índice"))
btn_indice.pack(pady=5)

ventana.mainloop()


Conexión establecida con el ESP32.
Modelo cargado exitosamente.
Scaler cargado exitosamente.


C:\Users\david\miniconda3\envs\entornoML2024\lib\site-packages\sklearn\base.py:347: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.0 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━

Exception in thread Thread-7:
Traceback (most recent call last):
  File "C:\Users\david\miniconda3\envs\entornoML2024\lib\threading.py", line 980, in _bootstrap_inner
    self.run()
  File "C:\Users\david\miniconda3\envs\entornoML2024\lib\threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\david\AppData\Local\Temp\ipykernel_19820\1344353721.py", line 55, in ejecutar_captura
  File "C:\Users\david\miniconda3\envs\entornoML2024\lib\site-packages\serial\serialwin32.py", line 275, in read
    raise SerialException("ClearCommError failed ({!r})".format(ctypes.WinError()))
serial.serialutil.SerialException: ClearCommError failed (PermissionError(13, 'Acceso denegado.', None, 5))
